# Module 10: Mortgage-Backed Securities (MBS) & Prepayment

Verification of prepayment models, pass-through math, and OAS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mbs.prepayment import psa_to_cpr, cpr_to_smm
from mbs.pass_through import project_cash_flows, average_life
from mbs.oas import calculate_oas, price_mbs

## 1. Prepayment Models (PSA to CPR/SMM)

In [ ]:
months = np.arange(1, 121)
cpr_100 = [psa_to_cpr(100, m) for m in months]
cpr_150 = [psa_to_cpr(150, m) for m in months]

plt.figure(figsize=(8, 4))
plt.plot(months, cpr_100, label="100% PSA")
plt.plot(months, cpr_150, label="150% PSA")
plt.xlabel("Month")
plt.ylabel("CPR")
plt.title("PSA Prepayment Curves")
plt.legend()
plt.show()

## 2. Pass-Through Cash Flows

In [ ]:
balance = 1_000_000
wac = 0.05
term = 360
smm_vector = [cpr_to_smm(psa_to_cpr(100, m)) for m in range(1, term + 1)]

cf = project_cash_flows(balance, wac, term, smm_vector)

plt.figure(figsize=(10, 5))
plt.plot(cf["scheduled_principal"], label="Scheduled Principal")
plt.plot(cf["prepayment"], label="Prepayment")
plt.plot(cf["interest"], label="Interest")
plt.xlabel("Month")
plt.ylabel("Cash Flow ($)")
plt.title("MBS Cash Flows (100% PSA)")
plt.legend()
plt.show()

al = average_life(cf["total_principal"], balance)
print(f"Average Life: {al:.2f} years")

## 3. Option-Adjusted Spread (OAS)

In [ ]:
n_paths = 10
np.random.seed(42)
rate_paths = np.random.normal(0.04, 0.005, (n_paths, term))

def prepay_model(r_path, wac):
    return np.clip((wac - r_path) * 2.0, 0, 0.2)

oas = calculate_oas(balance, balance, wac, term, rate_paths, prepay_model)
print(f"OAS for par price: {oas * 10000:.1f} bps")